<a href="https://colab.research.google.com/github/kiran92345/Ai-health-assistant/blob/main/MILESTONE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load both datasets
data1 = pd.read_csv("/content/meetinglistdetails_2026_05_29_2026_05_29.csv")
data2 = pd.read_csv("/content/meetinglistdetails_2026_05_30_2026_05_30.csv")

# Combine datasets
data = pd.concat([data1, data2], ignore_index=True)

# Clean column names
data.columns = data.columns.str.strip()

# Convert duration to numeric
data['Duration (minutes).1'] = pd.to_numeric(
    data['Duration (minutes).1'],
    errors='coerce'
)

# Present if attended 90 minutes or more
data['Present'] = (data['Duration (minutes).1'] >= 90).astype(int)

# Group by Email
attendance = data.groupby('Email').agg(
    Present_Sessions=('Present', 'sum'),
    Total_Records=('Present', 'count')
).reset_index()

# Total sessions
TOTAL_SESSIONS = 2

attendance['Attendance_Percentage'] = (
    attendance['Present_Sessions'] / TOTAL_SESSIONS
) * 100

# Eligibility
attendance['Eligible'] = attendance['Attendance_Percentage'].apply(
    lambda x: "Valid" if x >= 80 else "Not Valid"
)

# Clean email column
attendance['Email'] = (
    attendance['Email']
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Attendance Search System")
print("Type 'exit' to quit.\n")

# Continuous search loop
while True:
    email = input("Enter Student Email: ").strip().lower()

    # Exit condition
    if email in ["exit", "quit", "q"]:
        print("Exiting program...")
        break

    student = attendance[attendance['Email'] == email]

    if not student.empty:
        print("\n===== STUDENT DETAILS =====")
        print("Email:", student.iloc[0]['Email'])
        print("Present Sessions:", student.iloc[0]['Present_Sessions'])
        print(
            "Attendance Percentage:",
            round(student.iloc[0]['Attendance_Percentage'], 2),
            "%"
        )
        print("Certificate Status:", student.iloc[0]['Eligible'])
    else:
        print("Student not found!")

    print("-" * 40)